In [ ]:
%pip install -qU pinecone pinecone-text pinecone-notebooks

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/docs/semantic-search.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/docs/semantic-search.ipynb)

# Hybrid Search  With Pinecone Vector database

In this walkthrough we will see how to use Pinecone for semantic search. To begin we must install the required prerequisite libraries:

# 1. Create Index --similar to table
# 2. Data Prep
# 3. Insert data into Index created in step 1
# 4. Perform Hybrid Search
# 5. ReRanker
# 6. Send the Retrieved chunks to LLM
# 7. Delete Index if not required.

---

🚨 _Note: the above `pip install` is formatted for Jupyter notebooks. If running elsewhere you may need to drop the `!`._

---

## 1. Creating an Index

Now the data is ready, we can set up our index to store it.

We begin by initializing our connection to Pinecone. To do this we need a [free API key](https://app.pinecone.io).

In [ ]:
import os

# # initialize connection to pinecone (orget API key at app.pinecone.io)
# if not os.environ.get("PINECONE_API_KEY"):
#     from pinecone_notebooks.colab import Authenticate
#     Authenticate()

In [ ]:
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder

print("Pinecone and BM25 installed successfully!")

Pinecone and BM25 installed successfully!


In [ ]:
from google.colab import userdata

In [ ]:
from pinecone import Pinecone
api_key = userdata.get('PINECONE_API_KEY')

# configure client
pc = Pinecone(api_key=api_key)
print("Connected to Pinecone successfully!")

Connected to Pinecone successfully!


Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [ ]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index called `semantic-search-fast`. It's important that we align the index `dimension` and `metric` parameters with those required by the `MiniLM-L6` model.

In [ ]:
index_name = "hybrid-search-syncf-index-obama-speech"

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

print("Existing Pinecone indexes:", existing_indexes)


if index_name in existing_indexes:
    print("Homework index already exists.")
else:
    print("Homework index does not exist. Ready to create.")



Existing Pinecone indexes: ['apple-10k-rag', 'chat-app-rag']
Homework index does not exist. Ready to create.


In [ ]:
from pinecone import ServerlessSpec
import time

index_name = "hybrid-search-syncf-index-obama-speech"

In [ ]:
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="dotproduct",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(2)

index = pc.Index(index_name)

print("Hybrid Search index created successfully!")
print(index.describe_index_stats())

Hybrid Search index created successfully!
DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='dotproduct', namespaces=0)


In [ ]:
import time

existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1536,  # dimensionality of minilm
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

## 2. Data Prep

In [ ]:
!ls -lh /content/

total 4.0K
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -qU langchain langchain_community langchain_openai

In [ ]:
# from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
# from langchain.text_splitter import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Load an example document
with open("/content/sotu_address_obama.txt") as f:
    state_of_the_union = f.read()

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)
documents = text_splitter.create_documents([state_of_the_union])

In [ ]:
len(documents)

89

In [ ]:
documents[1]

Document(metadata={}, page_content='Our Constitution declares that from time to time, the President shall give to Congress information about the state of our union. For two hundred and twenty years, our leaders have fulfilled this duty. They have done so during periods of prosperity and tranquility. And they have done so in the midst of war and depression; at moments of great strife and great struggle.')

In [ ]:
# raw_documents = TextLoader('/content/sotu_address_obama.txt').load()
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
# documents = text_splitter.split_text(raw_documents[0].page_content)

In [ ]:
import os
#import getpass
#os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key:')
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [ ]:
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')

In [ ]:
!pip install --upgrade transformers

In [ ]:
from pinecone_text.sparse import BM25Encoder   # TFIDF, BOW, ONE HOT ENCODING(OHE)
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
bm25 = BM25Encoder.default()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
len(documents)

89

In [ ]:
documents

[Document(metadata={}, page_content='Barack Hussein Obama\nState of the Union 2010 - 27 January 2010\nMadame Speaker, Vice President Biden, Members of Congress, distinguished guests, and fellow Americans:'),
 Document(metadata={}, page_content='Our Constitution declares that from time to time, the President shall give to Congress information about the state of our union. For two hundred and twenty years, our leaders have fulfilled this duty. They have done so during periods of prosperity and tranquility. And they have done so in the midst of war and depression; at moments of great strife and great struggle.'),
 Document(metadata={}, page_content='It’s tempting to look back on these moments and assume that our progress was inevitable – that America was always destined to succeed. But when the Union was turned back at Bull Run and the Allies first landed at Omaha Beach, victory was very much in doubt. When the market crashed on Black Tuesday and civil rights marchers were beaten on Blood

In [ ]:
for chunk in documents[0:2]:
  print(chunk.page_content)
  dense_values=embeddings.embed_query(chunk.page_content)
  sparse_values = bm25.encode_queries(chunk.page_content)
  print(dense_values)
  print(sparse_values)

Barack Hussein Obama
State of the Union 2010 - 27 January 2010
Madame Speaker, Vice President Biden, Members of Congress, distinguished guests, and fellow Americans:
[-0.03155992925167084, -0.027338538318872452, -0.014247197657823563, -0.005053734872490168, -0.013317486271262169, 0.00481188390403986, -0.006168759893625975, 0.0036811542231589556, -0.012010864913463593, -0.0037816634867340326, 0.02645908109843731, -0.00379736814647913, -0.012048555538058281, -0.006470288150012493, 0.010572325438261032, -0.011872664093971252, 0.022664854303002357, -0.026408826932311058, 0.017086585983633995, -0.008392528630793095, 0.01838064379990101, -0.019963664934039116, 0.014661798253655434, 0.02390865609049797, -0.01633276604115963, -0.03691205009818077, 0.029172832146286964, -0.03256502375006676, 0.01756400614976883, -0.024599658325314522, 0.012670457363128662, -0.017991170287132263, -0.035429537296295166, -0.004337605554610491, -0.020101865753531456, -0.007393717300146818, -0.014611544087529182, -0

In [ ]:
from importlib import metadata
docs=[]
docs_json={}
id=0
for chunk in documents:
  id+=1
  dense_values=embeddings.embed_query(chunk.page_content)
  sparse_values = bm25.encode_queries(chunk.page_content)
  docs_json={
      'id':str(id),
      'values':dense_values,
      'sparse_values':sparse_values,
      'metadata':{
      'text':chunk.page_content,
      'source':"/content/sotu_address_obama.txt",
      'author':'zion',
      'createdDate':'2024-12-15'
      }
  }
  docs.append(docs_json)

In [ ]:
len(docs)

89

In [ ]:
len(docs[1]['values'])

1536

In [ ]:
docs[2]['id']

'3'

# 3. Insert data into Index




In [ ]:
# prompt: give me a code to process python list of batch size of 10
batch_size = 10
for i in range(0, len(docs), batch_size):
    batch = docs[i:i + batch_size]
    print(f"upserting batch {i} to {i + batch_size}")
    print(batch)
    index.upsert(batch)

Streaming output truncated to the last 5000 lines.
             3312546594],
 'values': [0.036703913243609956,
            0.0118362660250061,
            0.03040233351701436,
            0.015638320013291573,
            0.021094438996340843,
            0.02827360767461595,
            0.023338856599659615,
            0.025000274041235688,
            0.02706413050671118,
            0.02479449607400844,
            0.046762456282559636,
            0.009608044060984998,
            0.016322855216419196,
            0.023491120726469954,
            0.019540956534508006,
            0.01906180967097579,
            0.01992499109833648,
            0.023086617621168357,
            0.015002060827550222,
            0.024824416243217032,
            0.05140684706824902,
            0.012050755112198433,
            0.014567375835946385,
            0.021854755227262337,
            0.01960697777282947,
            0.011213250281705143,
            0.017874553259463702,
            0.0

## 4. Making Queries - Semantic Search

Now that our index is populated we can begin making queries. We are performing a semantic search for *similar questions*, so we should embed and search with another question. Let's begin.

Now let's query.

In [ ]:
# prompt ='''Summarize what was obama said about schools based on {context}'''.format(context=text))
# print(prompt)

In [ ]:
query = "summarize was obama said about schools?"

# create the query vector
#xq = model.encode(query).tolist()
dense_vector = embeddings.embed_query(query)
keyword_vector = bm25.encode_queries(query)

In [ ]:
# xc = index.query(vector=dense_vector,top_k=3, include_metadata=True)
# xc

In [ ]:
# now query
xc = index.query(vector=dense_vector, sparse_vector=keyword_vector,top_k=3, include_metadata=True)
xc

{'matches': [{'id': '42',
              'metadata': {'author': 'zion',
                           'createdDate': '2024-12-15',
                           'source': '/content/sotu_address_obama.txt',
                           'text': 'This year, we have broken through the '
                                   'stalemate between left and right by '
                                   'launching a national competition to '
                                   'improve our schools. The idea here is '
                                   'simple: instead of rewarding failure, we '
                                   'only reward success. Instead of funding '
                                   'the status quo, we only invest in reform – '
                                   'reform that raises student achievement, '
                                   'inspires students to excel in math and '
                                   'science, and turns around failing schools '
                            

In the returned response `xc` we can see the most relevant questions to our particular query — we don't have any exact matches but we can see that the returned questions are similar in the topics they are asking about. We can reformat this response to be a little easier to read:

# 5. Validate with CrossEncoder MODEL.
# Refer the Noetebook:CrossEncoder_ReRannking.ipynb


In [ ]:
# !pip install sentence_transformers -qU

In [ ]:
# from sentence_transformers import CrossEncoder
# model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=1024)

# query='what was obama said about schools'
# ans1=f'''Fourth, we need to invest in the skills and education of our people.
# This year, we have broken through the stalemate between left and right by launching a national competition to improve our schools. The idea here is simple: instead of rewarding failure, we only reward success. Instead of funding the status quo, we only invest in reform – reform that raises student achievement, inspires students to excel in math and science, and turns around failing schools that steal the future of too many young Americans, from rural communities to inner-cities. In the 21st century, one of the best anti-poverty programs is a world-class education. In this country, the success of our children cannot depend more on where they live than their potential.
# '''
# ans2=f'''
# When we renew the Elementary and Secondary Education Act, we will work with Congress to expand these reforms to all fifty states. Still, in this economy, a high school diploma no longer guarantees a good job. I urge the Senate to follow the House and pass a bill that will revitalize our community colleges, which are a career pathway to the children of so many working families. To make college more affordable, this bill will finally end the unwarranted taxpayer-subsidies that go to banks for student loans. Instead, let’s take that money and give families a $10,000 tax credit for four years of college and increase Pell Grants. And let’s tell another one million students that when they graduate, they will be required to pay only ten percent of their income on student loans, and all of their debt will be forgiven after twenty years – and forgiven after ten years if they choose a career in public service. Because in the United States of America, no one should go broke because they chose to go to college. And it’s time for colleges and universities to get serious about cutting their own costs – because they too have a responsibility to help solve this problem.
# '''
# ans3='''
# So I know the anxieties that are out there right now. They’re not new. These struggles are the reason I ran for President. These struggles are what I’ve witnessed for years in places like Elkhart, Indiana and Galesburg, Illinois. I hear about them in the letters that I read each night. The toughest to read are those written by children – asking why they have to move from their home, or when their mom or dad will be able to go back to work.
# For these Americans and so many others, change has not come fast enough. Some are frustrated; some are angry. They don’t understand why it seems like bad behavior on Wall Street is rewarded but hard work on Main Street isn’t; or why Washington has been unable or unwilling to solve any of our problems. They are tired of the partisanship and the shouting and the pettiness. They know we can’t afford it. Not now.
# '''
# scores = model.predict([(query,ans1), (query,ans2),(query,ans3)])

In [ ]:
#
# array([ -6.0359206,  -6.9418993, -10.022743 ], dtype=float32)

# 6. Send the Retrieved chunks to LLM

In [ ]:
context=''
for result in xc['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['text']}")
    context=context + '\n'+ result['metadata']['text']

0.82: This year, we have broken through the stalemate between left and right by launching a national competition to improve our schools. The idea here is simple: instead of rewarding failure, we only reward success. Instead of funding the status quo, we only invest in reform – reform that raises student achievement, inspires students to excel in math and science, and turns around failing schools that steal the future of too many young Americans, from rural communities to inner-cities. In the 21st century, one of the best anti-poverty programs is a world-class education. In this country, the success of our children cannot depend more on where they live than their potential.
0.81: Barack Hussein Obama
State of the Union 2010 - 27 January 2010
Madame Speaker, Vice President Biden, Members of Congress, distinguished guests, and fellow Americans:
0.81: When we renew the Elementary and Secondary Education Act, we will work with Congress to expand these reforms to all fifty states. Still, in 

In [ ]:
print(context)


This year, we have broken through the stalemate between left and right by launching a national competition to improve our schools. The idea here is simple: instead of rewarding failure, we only reward success. Instead of funding the status quo, we only invest in reform – reform that raises student achievement, inspires students to excel in math and science, and turns around failing schools that steal the future of too many young Americans, from rural communities to inner-cities. In the 21st century, one of the best anti-poverty programs is a world-class education. In this country, the success of our children cannot depend more on where they live than their potential.
Barack Hussein Obama
State of the Union 2010 - 27 January 2010
Madame Speaker, Vice President Biden, Members of Congress, distinguished guests, and fellow Americans:
When we renew the Elementary and Secondary Education Act, we will work with Congress to expand these reforms to all fifty states. Still, in this economy, a h

These are good results, let's try and modify the words being used to see if we still surface similar results.

In [ ]:
prompt='''
Anwer the following question based on context
question: {query}
context: {context}
'''.format(context=context,query=query)
print(prompt)


Anwer the following question based on context
question: summarize was obama said about schools?
context: 
This year, we have broken through the stalemate between left and right by launching a national competition to improve our schools. The idea here is simple: instead of rewarding failure, we only reward success. Instead of funding the status quo, we only invest in reform – reform that raises student achievement, inspires students to excel in math and science, and turns around failing schools that steal the future of too many young Americans, from rural communities to inner-cities. In the 21st century, one of the best anti-poverty programs is a world-class education. In this country, the success of our children cannot depend more on where they live than their potential.
Barack Hussein Obama
State of the Union 2010 - 27 January 2010
Madame Speaker, Vice President Biden, Members of Congress, distinguished guests, and fellow Americans:
When we renew the Elementary and Secondary Educatio

In [ ]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [ ]:
from langchain_groq import ChatGroq
chat_model=ChatGroq(model='openai/gpt-oss-20b',temperature=.5,max_tokens=1000)

In [ ]:
import pprint

In [ ]:
response=chat_model.invoke(prompt)
pprint.pprint(response.content)

('In his 2010 State of the Union, President\u202fObama called for a national '
 '“competition” to raise school quality by rewarding success rather than '
 'maintaining the status quo. He urged reforms that boost student achievement, '
 'especially in math and science, and turn around failing schools across the '
 'country. Obama framed a world‑class education as a top anti‑poverty strategy '
 'and stressed that a high‑school diploma alone no longer guarantees a good '
 'job. He pushed for a renewed Elementary and Secondary Education Act, '
 'expansion of community‑college pathways, and a package of measures to make '
 'college affordable—including tax credits, larger Pell Grants, and a plan to '
 'forgive student debt after a set period or for public‑service careers.')


In [ ]:
print(response.content)

In his 2010 State of the Union, President Obama called for a national “competition” to raise school quality by rewarding success rather than maintaining the status quo. He urged reforms that boost student achievement, especially in math and science, and turn around failing schools across the country. Obama framed a world‑class education as a top anti‑poverty strategy and stressed that a high‑school diploma alone no longer guarantees a good job. He pushed for a renewed Elementary and Secondary Education Act, expansion of community‑college pathways, and a package of measures to make college affordable—including tax credits, larger Pell Grants, and a plan to forgive student debt after a set period or for public‑service careers.


# 7. Delete Index if not required

In [ ]:
#index.delete(delete_all=True, index_name=index_name)